#　DermMelデータセットでResNet50分類器の訓練

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score
import pandas as pd
from PIL import Image
import numpy as np
import os
from pathlib import Path
import random

# =========================
# 設定
# =========================
DATA_DIR = Path("/mnt/data1/Public/MedImages/DermMel")
TRAIN_IMG_DIR = DATA_DIR / "train_sep"
VAL_IMG_DIR = DATA_DIR / "valid"
TEST_IMG_DIR = DATA_DIR / "test"

# DermMel はフォルダ名がクラス名になっている構造を想定
# train_sep/
# ├── Melanoma/
# └── NotMelanoma/
# valid/
# ├── Melanoma/
# └── NotMelanoma/
# test/（ラベルなし想定）

# =========================
# データ変換（Data Augmentation）
# =========================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =========================
# Dataset定義
# =========================
class ImageFolderWithLabel(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {"NotMelanoma": 0, "Melanoma": 1}

        for cls in self.class_to_idx.keys():
            class_dir = Path(root_dir) / cls
            files = list(class_dir.glob("*.*"))
            for f in files:
                self.samples.append((f, self.class_to_idx[cls]))

        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# =========================
# Dataset & DataLoader
# =========================
train_dataset = ImageFolderWithLabel(TRAIN_IMG_DIR, train_transform)
val_dataset = ImageFolderWithLabel(VAL_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# =========================
# モデル定義
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)  # 2クラス→1出力 (sigmoid)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# =========================
# EarlyStopping（モデル保存機能付き）
# =========================
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, save_path="best_model.pth"):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta
        self.best_model = None
        self.save_path = save_path  # 追加

    def __call__(self, val_acc, model):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.best_model = model.state_dict()
            torch.save(self.best_model, self.save_path)
            if self.verbose:
                print(f"✅ Best model saved (acc: {val_acc:.4f})")
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model = model.state_dict()
            torch.save(self.best_model, self.save_path)
            if self.verbose:
                print(f"✅ Best model updated (acc: {val_acc:.4f})")
            self.counter = 0



In [2]:

# =========================
# 学習ループ
# =========================
early_stopping = EarlyStopping(patience=7, verbose=True, save_path="DermMel_best_resnet50.pth")
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

    # 検証
    model.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images).squeeze()
            preds = torch.sigmoid(outputs) > 0.5
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Val Accuracy: {val_acc*100:.2f}%")

    early_stopping(val_acc, model)
    if early_stopping.early_stop:
        print("Early stopping!")
        break

# 最良モデル復元
model.load_state_dict(early_stopping.best_model)

# =========================
# テストデータ予測（TTAあり）
# =========================
test_files = list(Path(TEST_IMG_DIR).glob("*.*"))
test_preds = []

model.eval()
for img_path in test_files:
    img = Image.open(img_path).convert("RGB")
    tensors = []
    for flip in [None, "H", "V"]:
        aug_img = img
        if flip == "H":
            aug_img = img.transpose(Image.FLIP_LEFT_RIGHT)
        elif flip == "V":
            aug_img = img.transpose(Image.FLIP_TOP_BOTTOM)
        tensor = val_transform(aug_img).unsqueeze(0).to(device)
        tensors.append(tensor)

    with torch.no_grad():
        outputs = [torch.sigmoid(model(t)).item() for t in tensors]
        pred = np.mean(outputs)
        test_preds.append(pred)

Epoch 1/30, Loss: 0.2138
Val Accuracy: 95.00%
✅ Best model saved (acc: 0.9500)
Epoch 2/30, Loss: 0.1360
Val Accuracy: 95.42%
✅ Best model updated (acc: 0.9542)
Epoch 3/30, Loss: 0.1120
Val Accuracy: 95.79%
✅ Best model updated (acc: 0.9579)
Epoch 4/30, Loss: 0.0989
Val Accuracy: 95.62%
EarlyStopping counter: 1 out of 7
Epoch 5/30, Loss: 0.0869
Val Accuracy: 95.09%
EarlyStopping counter: 2 out of 7
Epoch 6/30, Loss: 0.0787
Val Accuracy: 95.45%
EarlyStopping counter: 3 out of 7
Epoch 7/30, Loss: 0.0647
Val Accuracy: 95.70%
EarlyStopping counter: 4 out of 7
Epoch 8/30, Loss: 0.0591
Val Accuracy: 95.31%
EarlyStopping counter: 5 out of 7
Epoch 9/30, Loss: 0.0506
Val Accuracy: 96.01%
✅ Best model updated (acc: 0.9601)
Epoch 10/30, Loss: 0.0482
Val Accuracy: 96.04%
✅ Best model updated (acc: 0.9604)
Epoch 11/30, Loss: 0.0414
Val Accuracy: 95.90%
EarlyStopping counter: 1 out of 7
Epoch 12/30, Loss: 0.0330
Val Accuracy: 96.18%
✅ Best model updated (acc: 0.9618)
Epoch 13/30, Loss: 0.0317
Val Acc